# Formulator-Pen manuscript: main figures 2-7

Regenerates Figures 2-7 of the manuscript from `../data/` and the artwork in `../figures/inputs/`, writing SVG and PNG files to `../figures/main/`. Run from this folder (`Manuscript/analysis`). Viscosities are the measured values of `../data/fluids.csv`. Figure 1 is product artwork (`../figures/main/fig1_formulator_pen.png`).


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch, FancyBboxPatch
from scipy import stats
from PIL import Image

## 2. Load Dataset

In [ ]:
data_path = "../data/formulatorpen.csv"
df = pd.read_csv(data_path)

# The CSV stores nominal grades; every figure and statistic uses the measured viscosities (Table 2)
VISC_MEASURED = {1: 1, 1000: 1000, 5000: 5765, 12500: 14570, 30000: 31870, 38000: 37020, 60000: 69080}
DENSITY = {1: 0.998, 1000: 1.261}   # g/mL; silicone oils default to 0.973
df["Viscosity (cP)"] = df["Viscosity (cP)"].map(VISC_MEASURED)
df.head()

## 3. Shared Style, Colors, and Helpers

In [ ]:
G_BLUE = "#4285F4"
G_DARK_BLUE = "#174EA6"
G_RED = "#EA4335"
G_YELLOW = "#FBBC04"
G_GREEN = "#34A853"
G_ORANGE = "#FF6D01"

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]
plt.rcParams["figure.dpi"] = 72   # inline preview only; files are saved at explicit dpi

run_cols = [f"Run{i}" for i in range(1, 11)]

mech_names   = {3: "Fill-and-Dispense", 4: "Remote-Reservoir", 5: "On-Board Reservoir", 6: "Pressurized On-Board"}
mech_colors  = {3: G_BLUE, 4: G_ORANGE, 5: G_GREEN, 6: G_RED}
mech_markers = {3: "o", 4: "s", 5: "^", 6: "D"}
mech_order   = [3, 4, 5, 6]

LABEL_FS  = 20
TICK_FS   = 15
LEGEND_FS = 13
PANEL_FS  = 22

def style_panel(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.6)
    ax.tick_params(axis="both", labelsize=TICK_FS, width=1.6, length=5)
    ax.xaxis.label.set_fontsize(LABEL_FS)
    ax.yaxis.label.set_fontsize(LABEL_FS)
    leg = ax.get_legend()
    if leg:
        for txt in leg.get_texts():
            txt.set_fontsize(LEGEND_FS)

def label_panels(fig, specs):
    """specs: list of (label, ax, row, col) -> places A/B/C.. above each panel, aligned by row/col."""
    row_tops, col_lefts = {}, {}
    for _, ax, r, c in specs:
        bb = ax.get_position()
        row_tops[r] = max(row_tops.get(r, 0), bb.y1)
        col_lefts[c] = min(col_lefts.get(c, 1), bb.x0)
    for label, ax, r, c in specs:
        color = "black"
        fig.text(col_lefts[c] - 0.045, row_tops[r] + 0.012, label, fontsize=PANEL_FS,
                  va="top", ha="left", fontfamily="Arial", fontweight="normal", color=color)

def prep_prototype(proto):
    d = df[df["Prototype"] == proto].copy()
    d[run_cols] = d[run_cols].apply(pd.to_numeric, errors="coerce")
    d["mean_dispensed"] = d[run_cols].mean(axis=1)
    d["std_dispensed"]  = d[run_cols].std(axis=1)
    d["n_runs"]         = d[run_cols].notna().sum(axis=1)
    d["cv_pct"]         = (d["std_dispensed"] / d["mean_dispensed"]) * 100
    d["error_pct"]      = ((d["mean_dispensed"] - d["Volume (g)"]) / d["Volume (g)"]) * 100
    return d

proto_data = {p: prep_prototype(p) for p in mech_order}

## Figure 3 - Operating sequence of the four liquid-handling strategies (Section 3.1)

Figure 2 is the externally supplied flowchart `figure2_flowchart.png` (step-by-step operation of each of the four strategies), not something plotted from the CSV. This cell only overlays the A-D column labels used everywhere else in the chapter, so `Fig. 2A`-style references stay consistent across figures.

In [ ]:
# Operating sequence of each strategy, drawn as vector shapes (header, step boxes, connectors)
flow_steps = {
    3: ["Move Tool to External\nReservoir", "Aspirate Liquid", "Move Tool to\nDestination", "Dispense Liquid"],
    4: ["Activate Secondary\nPump", "Prime Tubes", "Aspirate via Pen\nMechanism", "Move Tool to\nDestination", "Dispense Liquid"],
    5: ["Aspirate from Attached\nReservoir", "Move Tool to\nDestination", "Dispense Liquid"],
    6: ["Pressurize Vessel", "Aspirate from Vessel", "Move Tool to\nDestination", "Dispense Liquid"],
}
flow_titles = {3: "Fill and Dispense", 4: "Remote Reservoir", 5: "On-Board Reservoir", 6: "Pressurized\nOn-Board"}
flow_colors = {  # (header fill, step fill, connector) sampled from the original flowchart
    3: ("#a9aae5", "#e2e3f6", "#6a6cd2"),
    4: ("#8eb4f8", "#d9e6fd", "#3c7df3"),
    5: ("#66c29a", "#ccebdd", "#02944c"),
    6: ("#ffd466", "#fff1cc", "#ffb500"),
}
panel_letters_row = {3: "A", 4: "B", 5: "C", 6: "D"}

# Layout in drawing units (1 unit ~ 1 px of the original 1086-px-wide flowchart)
box_w, col_pitch, pad = 257, 285, 12
hdr_y, hdr_h = 52, 75
step_y0, step_h, step_pitch = 160, 58, 79
n_max = max(len(s) for s in flow_steps.values())
total_w = col_pitch * len(mech_order) - (col_pitch - box_w - 2 * pad)
total_h = step_y0 + (n_max - 1) * step_pitch + step_h + pad + 4

fig2, ax2 = plt.subplots(figsize=(14.5, 14.5 * total_h / total_w))
fig2.subplots_adjust(0, 0, 1, 1)
ax2.set_xlim(0, total_w)
ax2.set_ylim(total_h, 0)
ax2.set_aspect("equal")
ax2.axis("off")

def rbox(x, y, w, h, fc, r=10):
    ax2.add_patch(FancyBboxPatch((x + r, y + r), w - 2 * r, h - 2 * r, boxstyle=f"round,pad={r}",
                                 facecolor=fc, edgecolor="none"))

for i, proto in enumerate(mech_order):
    cx0 = i * col_pitch
    hdr_fc, step_fc, line_c = flow_colors[proto]
    ax2.text(cx0 + pad, 12, panel_letters_row[proto], fontsize=PANEL_FS, va="top", ha="left", color="black")

    x = cx0 + pad
    rbox(x, hdr_y, box_w, hdr_h, hdr_fc)
    ax2.text(x + box_w / 2, hdr_y + hdr_h / 2, flow_titles[proto], fontsize=19.5, ha="center", va="center", linespacing=1.1)

    steps = flow_steps[proto]
    for k, step in enumerate(steps):
        y = step_y0 + k * step_pitch
        rbox(x, y, box_w, step_h, step_fc, r=8)
        ax2.text(x + box_w / 2, y + step_h / 2, step, fontsize=17, ha="center", va="center", linespacing=1.1)
        if k < len(steps) - 1:
            ax2.plot([x + box_w / 2] * 2, [y + step_h, y + step_pitch], color=line_c, linewidth=2.5, solid_capstyle="butt")

plt.savefig("../figures/main/fig3_strategy_sequences.svg", format="svg", dpi=300)
plt.show()
print("Saved: figure2_flowchart_labeled.svg")

## Figure 2 - Orchestration workflow and platform (Section 2.5)

Both panels are drawn as vector shapes (SVG for the manuscript source, 600-dpi PNG for Word): (A) the nested dispensing workflow and (B) the workcell architecture with its communication links (USB/RS-232, Tailscale VPN, WiFi to the pens) and the mechanical tool exchange.

In [ ]:
# Figure 2 (orchestration), fully vector. Panel A: nested dispensing workflow. Panel B: workcell
# architecture and communication links. Saved as SVG (manuscript source) and 600-dpi PNG (Word).
from matplotlib.patches import FancyBboxPatch, Rectangle, Polygon
from matplotlib.lines import Line2D

NAVY, INK = "#2F4B7C", "#1E2533"
STEP_FC, STEP_EC = "#F3F7FC", NAVY
TERM_FC, TERM_EC = "#E5F3EA", "#2E7D4F"
DEC_FC, DEC_EC = "#FFF4DE", "#B7791F"
LOOP1_FC, LOOP2_FC, LOOP_EC = "#E9F0F8", "#F8FAFD", "#7C93B5"
C_USB, C_VPN, C_WIFI, C_MECH = NAVY, "#D9822B", "#1F9E89", "#6B6B6B"
FS = 15

fig2o = plt.figure(figsize=(13, 14.2))
axA = fig2o.add_axes([0.0, 0.535, 1.0, 0.465])
axB = fig2o.add_axes([0.0, 0.0, 1.0, 0.515])
for ax, h in ((axA, 6.55), (axB, 7.05)):
    ax.set_xlim(0, 13); ax.set_ylim(h, 0); ax.axis("off")

def box(ax, x, y, w, h, text, fc, ec, fs=FS, color=INK, lw=1.5, r=0.10, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}",
                                facecolor=fc, edgecolor=ec, linewidth=lw, zorder=3))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, color=color,
            linespacing=1.12, zorder=4, fontweight="bold" if bold else "normal")

def diamond(ax, cx, cy, w, h, text):
    ax.add_patch(Polygon([(cx - w / 2, cy), (cx, cy - h / 2), (cx + w / 2, cy), (cx, cy + h / 2)],
                         closed=True, facecolor=DEC_FC, edgecolor=DEC_EC, linewidth=1.5, zorder=3))
    ax.text(cx, cy, text, ha="center", va="center", fontsize=FS - 1.5, color=INK, linespacing=1.05, zorder=4)

def path(ax, pts, color=INK, ls="-", lw=1.5, head=True, both=False, label=None, lxy=None, lcolor=None, lfs=FS - 2):
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, linestyle=ls, linewidth=lw, zorder=2, solid_capstyle="butt")
    def head_at(tip, tail):
        # arrowhead only: a short solid stub so dashed and dotted line styles stay visible
        (x1, y1), (x0, y0) = tip, tail
        L = ((x1 - x0) ** 2 + (y1 - y0) ** 2) ** 0.5 or 1.0
        stub = min(0.12, L)
        ax.annotate("", xy=tip, xytext=(x1 - (x1 - x0) / L * stub, y1 - (y1 - y0) / L * stub), zorder=2,
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=16, shrinkA=0, shrinkB=0))
    if head or both:
        head_at(pts[-1], pts[-2])
    if both:
        head_at(pts[0], pts[1])
    if label:
        ax.text(*lxy, label, fontsize=lfs, ha="center", va="center", color=lcolor or color, zorder=5,
                bbox=dict(facecolor="white", edgecolor="none", pad=1.5, alpha=0.9))

def loop(ax, x, y, w, h, title, fc):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.14", facecolor=fc,
                                edgecolor=LOOP_EC, linewidth=1.4, linestyle=(0, (5, 3)), zorder=1))
    ax.text(x + 0.12, y + h - 0.1, title, fontsize=FS - 0.5, fontweight="bold", color=NAVY, ha="left", va="bottom", zorder=4)

# ------------------------------------------------------------------ panel A: workflow
axA.text(0.02, 0.02, "A", fontsize=21, ha="left", va="top", color="black")
bw, bh = 2.35, 0.85
box(axA, 0.15, 0.75, bw, bh, "Input recipe via\ncustom dashboard", TERM_FC, TERM_EC)
box(axA, 0.15, 2.15, bw, bh, "Generate dispensing\nsequence and\ngantry path", STEP_FC, STEP_EC)
path(axA, [(0.15 + bw / 2, 0.75 + bh), (0.15 + bw / 2, 2.15)])

loop(axA, 2.85, 0.45, 7.6, 6.0, "For each material", LOOP1_FC)
box(axA, 3.05, 0.95, 2.2, bh, "Select next\ndispenser (≤5;\n2 in this study)", STEP_FC, STEP_EC)
box(axA, 3.05, 2.35, 2.2, bh, "Pick up dispenser;\nprime if required", STEP_FC, STEP_EC)
path(axA, [(4.15, 0.95 + bh), (4.15, 2.35)])
path(axA, [(0.15 + bw, 2.15 + bh / 2), (2.7, 2.15 + bh / 2), (2.7, 0.95 + bh / 2), (3.05, 0.95 + bh / 2)])

loop(axA, 5.55, 0.85, 2.75, 5.35, "For each container", LOOP2_FC)
cx0, cw = 5.7, 2.45
mid = cx0 + cw / 2
box(axA, cx0, 1.3, cw, 0.62, "Move to next container", STEP_FC, STEP_EC)
box(axA, cx0, 2.17, cw, 0.62, "Tare balance", STEP_FC, STEP_EC)
box(axA, cx0, 3.04, cw, 0.85, "Dispense calibrated\ndose (feed-forward)", STEP_FC, STEP_EC)
box(axA, cx0, 4.14, cw, 0.62, "Wait 10 s; record mass", STEP_FC, STEP_EC)
diamond(axA, mid, 5.45, 2.3, 0.95, "More\ncontainers?")
for y0, y1 in ((1.92, 2.17), (2.79, 3.04), (3.89, 4.14), (4.76, 4.975)):
    path(axA, [(mid, y0), (mid, y1)])
path(axA, [(5.25, 2.35 + bh / 2), (5.4, 2.35 + bh / 2), (5.4, 1.61), (cx0, 1.61)])
path(axA, [(mid - 1.15, 5.45), (5.62, 5.45), (5.62, 1.12), (mid, 1.12), (mid, 1.3)], label="Yes", lxy=(5.88, 5.2))

rx, rw = 8.45, 1.85
rcx = rx + rw / 2
box(axA, rx, 5.03, rw, bh, "Return dispenser\nto tool dock", STEP_FC, STEP_EC)
diamond(axA, rcx, 3.3, 1.95, 1.05, "More\nmaterials?")
path(axA, [(mid + 1.15, 5.45), (rx, 5.45)], label="No", lxy=(8.12, 5.2))
path(axA, [(rcx, 5.03), (rcx, 3.825)])
path(axA, [(rcx, 2.775), (rcx, 0.68), (4.15, 0.68), (4.15, 0.95)], label="Yes", lxy=(rcx + 0.32, 2.45))

fx = 10.75
box(axA, fx, 2.875, 2.1, bh, "Return gantry\nto home position", STEP_FC, STEP_EC)
box(axA, fx, 4.4, 2.1, 0.85, "Workflow\ncomplete", TERM_FC, TERM_EC)
path(axA, [(rcx + 0.975, 3.3), (fx, 3.3)], label="No", lxy=(10.6, 3.02))
path(axA, [(fx + 1.05, 2.875 + bh), (fx + 1.05, 4.4)])

# ------------------------------------------------------------------ panel B: architecture
axB.text(0.02, 0.05, "B", fontsize=21, ha="left", va="top", color="black")
axB.add_patch(FancyBboxPatch((3.2, 0.3), 9.65, 5.95, boxstyle="round,pad=0,rounding_size=0.25",
                             facecolor="#FBFCFE", edgecolor="#9AA9C1", linewidth=1.6, zorder=0))
axB.text(12.72, 6.12, "Formulation workcell", fontsize=FS - 1, color="#5B6B85", ha="right", va="bottom", style="italic")

def syringe(ax, x, y, s=0.62, body="#FFFFFF", edge=INK):
    """Small pen glyph: plunger, barrel with fluid, valve, needle (top-left anchored)."""
    ax.add_patch(Rectangle((x + 0.11 * s, y), 0.10 * s, 0.18 * s, facecolor=edge, edgecolor=edge, zorder=5))
    ax.add_patch(Rectangle((x, y + 0.18 * s), 0.32 * s, 0.62 * s, facecolor=body, edgecolor=edge, lw=1.2, zorder=5))
    ax.add_patch(Rectangle((x + 0.03 * s, y + 0.45 * s), 0.26 * s, 0.33 * s, facecolor="#7B8FD6", edgecolor="none", zorder=6))
    ax.add_patch(Rectangle((x + 0.08 * s, y + 0.80 * s), 0.16 * s, 0.12 * s, facecolor="#F28E2B", edgecolor=edge, lw=1.0, zorder=5))
    ax.plot([x + 0.16 * s] * 2, [y + 0.92 * s, y + 1.22 * s], color=edge, lw=1.4, zorder=5)

# tool dock with pens and free slots
axB.add_patch(FancyBboxPatch((3.5, 0.62), 9.05, 1.62, boxstyle="round,pad=0,rounding_size=0.12", facecolor="#F4F6FA",
                             edgecolor=LOOP_EC, linewidth=1.4, linestyle=(0, (5, 3)), zorder=1))
axB.text(3.62, 0.70, "Tool dock: interchangeable, material-specific dispensers (up to five)", fontsize=FS - 1.5, color=NAVY,
         fontweight="bold", ha="left", va="top")
box(axB, 3.75, 1.12, 3.15, 0.95, "      Formulator-Pen 1 (white)\n      Bluesilv12, 14,570 cP", "#FFFFFF", INK, fs=FS - 2)
syringe(axB, 3.9, 1.18, s=0.62)
box(axB, 7.15, 1.12, 3.15, 0.95, "      Formulator-Pen 2 (black)\n      Siltech60, 37,020 cP", "#2B2F36", INK, fs=FS - 2, color="white")
syringe(axB, 7.3, 1.18, s=0.62, body="#DDE2EA", edge="#FFFFFF")
for k in range(3):
    axB.add_patch(FancyBboxPatch((10.55 + k * 0.66, 1.12), 0.52, 0.72, boxstyle="round,pad=0,rounding_size=0.08",
                                 facecolor="none", edgecolor="#A9B4C6", linewidth=1.2, linestyle=(0, (3, 2)), zorder=3))
axB.text(11.54, 1.95, "free slots", fontsize=FS - 3.5, color="#7A869A", ha="center", va="center")

# control PC, hub, balances, gantry
box(axB, 0.15, 2.55, 2.7, 1.45, "Control PC\nMADSci orchestrator\n+ recipe dashboard", "#E9F0F8", NAVY, fs=FS - 1.5)
box(axB, 3.6, 4.85, 4.35, 1.1, "Raspberry Pi 5 device hub\nSiLA2 servers (one per device)", "#E9F0F8", NAVY, fs=FS - 1.5)
box(axB, 3.6, 2.75, 1.95, 1.0, "Balance 1\n3 vials", "#F1F2F4", "#7A7F87", fs=FS - 2)
box(axB, 6.0, 2.75, 1.95, 1.0, "Balance 2\n3 vials", "#F1F2F4", "#7A7F87", fs=FS - 2)
box(axB, 9.05, 3.05, 3.55, 2.65, "XYZ gantry\n(Genmitsu 4040-PRO)\n\n+ Jubilee\ntool changer", "#FFF4DE", "#B7791F", fs=FS - 1.5)

# links
path(axB, [(1.5, 4.0), (1.5, 5.4), (3.6, 5.4)], color=C_VPN, ls=(0, (6, 3)), lw=1.9, both=True,
     label="Tailscale VPN", lxy=(1.5, 4.75))
path(axB, [(4.575, 4.85), (4.575, 3.75)], color=C_USB, lw=1.9, both=True)
path(axB, [(6.975, 4.85), (6.975, 3.75)], color=C_USB, lw=1.9, both=True)
axB.text(5.78, 4.3, "USB / RS-232", fontsize=FS - 2, color=C_USB, ha="center", va="center",
         bbox=dict(facecolor="white", edgecolor="none", pad=1.5))
path(axB, [(7.95, 5.55), (9.05, 5.55)], color=C_USB, lw=1.9, both=True, label="USB", lxy=(8.5, 5.3))
path(axB, [(7.95, 5.05), (8.55, 5.05), (8.55, 2.24)], color=C_WIFI, ls=(0, (1.5, 2.2)), lw=2.4, both=True,
     label="WiFi\n(pen Pico W)", lxy=(8.55, 3.35))
path(axB, [(10.83, 3.05), (10.83, 2.24)], color=C_MECH, lw=2.2, both=True)
axB.text(10.98, 2.66, "pick-up / return;\npower via magnetic\nconnector", fontsize=FS - 3.5, color=C_MECH, ha="left", va="center")

# legend
handles = [Line2D([0], [0], color=C_USB, lw=2, label="USB / RS-232"),
           Line2D([0], [0], color=C_VPN, lw=2, ls=(0, (6, 3)), label="Tailscale VPN"),
           Line2D([0], [0], color=C_WIFI, lw=2.4, ls=(0, (1.5, 2.2)), label="WiFi"),
           Line2D([0], [0], color=C_MECH, lw=2.2, label="Mechanical tool exchange")]
axB.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.005), ncol=4, frameon=False, fontsize=FS - 2,
           handlelength=2.6, columnspacing=1.6)

plt.savefig("../figures/main/fig2_orchestration.svg", format="svg", bbox_inches="tight")
plt.savefig("../figures/main/fig2_orchestration.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figure2_orchestration_v2.svg / .png")


## Figure 4 - Dispensing accuracy (Section 3.2)

In [ ]:
def regression_by_fluid(proto):
    d = proto_data[proto]
    cal = d[d["Data Type"] == "Calibration"]
    out = []
    for fluid, visc in cal[["Fluid", "Viscosity (cP)"]].drop_duplicates().values:
        sub = cal[cal["Fluid"] == fluid].sort_values("Volume (g)")
        if len(sub) < 2:
            continue
        slope, intercept, r, _, _ = stats.linregress(sub["Volume (g)"], sub["mean_dispensed"])
        residuals = sub["mean_dispensed"] - (slope * sub["Volume (g)"] + intercept)
        rmse = np.sqrt((residuals ** 2).mean())
        out.append((fluid, visc, slope, rmse))
    return sorted(out, key=lambda x: x[1])

def metrics_at_target(proto, target):
    d = proto_data[proto]
    test = d[(d["Data Type"] == "Test") & (np.isclose(d["Volume (g)"], target))]
    rows = []
    for fluid, visc in test[["Fluid", "Viscosity (cP)"]].drop_duplicates().values:
        sub = test[test["Fluid"] == fluid]
        if sub.empty:
            continue
        rows.append((fluid, visc, sub["error_pct"].mean(), sub["cv_pct"].mean()))
    return sorted(rows, key=lambda x: x[1])

reg_by_proto = {p: regression_by_fluid(p) for p in mech_order}
metrics3g = {p: metrics_at_target(p, 3.0) for p in mech_order}

fig3 = plt.figure(figsize=(21, 12))
gs3 = gridspec.GridSpec(2, 6, figure=fig3, hspace=0.32, wspace=0.55)
ax3A = fig3.add_subplot(gs3[0, 0:2])
ax3B = fig3.add_subplot(gs3[0, 2:4])
ax3C = fig3.add_subplot(gs3[0, 4:6])
ax3D = fig3.add_subplot(gs3[1, 1:3])
ax3E = fig3.add_subplot(gs3[1, 3:5])

# A: calibration curves overlay at the shared 12,500 cP condition
max_target_A = 0
for proto in mech_order:
    d = proto_data[proto]
    cal = d[(d["Data Type"] == "Calibration") & (d["Fluid"] == "Bluesil_47V12500")].sort_values("Volume (g)")
    if cal.empty:
        continue
    max_target_A = max(max_target_A, cal["Volume (g)"].max())
    ax3A.errorbar(cal["Volume (g)"], cal["mean_dispensed"], yerr=cal["std_dispensed"],
                  fmt=mech_markers[proto] + "-", color=mech_colors[proto], label=mech_names[proto],
                  capsize=4, linewidth=2.2, markersize=8, elinewidth=1.3)
lims3A = [0, max_target_A * 1.05]
ax3A.plot(lims3A, lims3A, "k--", linewidth=1.5, alpha=0.55, label="Ideal")
ax3A.set_xlim(lims3A); ax3A.set_ylim(lims3A)
ax3A.set_xlabel("Target Mass (g)"); ax3A.set_ylabel("Dispensed Mass (g)")
ax3A.legend(fontsize=LEGEND_FS, frameon=False)
ax3A.grid(False)
style_panel(ax3A)

# B: slope vs viscosity, one line per strategy
for proto in mech_order:
    reg = reg_by_proto[proto]
    if not reg:
        continue
    viscs = [r[1] for r in reg]
    slopes = [r[2] for r in reg]
    ax3B.plot(viscs, slopes, mech_markers[proto] + "-", color=mech_colors[proto], label=mech_names[proto], linewidth=2.2, markersize=9)
ax3B.axhline(1.0, color="black", linewidth=1.5, linestyle="--", alpha=0.8)
ax3B.set_xscale("log")
ax3B.set_xlabel("Viscosity (cP)"); ax3B.set_ylabel("Pre-Calibration Slope")
ax3B.grid(False)
style_panel(ax3B)

# C: RMSE vs viscosity, one line per strategy
for proto in mech_order:
    reg = reg_by_proto[proto]
    if not reg:
        continue
    viscs = [r[1] for r in reg]
    rmses = [r[3] for r in reg]
    ax3C.plot(viscs, rmses, mech_markers[proto] + "-", color=mech_colors[proto], label=mech_names[proto], linewidth=2.2, markersize=9)
ax3C.set_xscale("log")
ax3C.set_xlabel("Viscosity (cP)"); ax3C.set_ylabel("Pre-Calibration RMSE (g)")
ax3C.grid(False)
style_panel(ax3C)

# D: post-calibration error vs viscosity at matched 3.0 g target
for proto in mech_order:
    rows = metrics3g[proto]
    if not rows:
        continue
    viscs = [r[1] for r in rows]; errs = [r[2] for r in rows]
    ax3D.plot(viscs, errs, mech_markers[proto] + "-", color=mech_colors[proto], label=mech_names[proto], linewidth=2.2, markersize=9)
ax3D.axhline(0, color="black", linewidth=1.3, linestyle="--", alpha=0.7)
ax3D.axhline(2, color=G_RED, linewidth=1.1, linestyle=":", alpha=0.7)
ax3D.axhline(-2, color=G_RED, linewidth=1.1, linestyle=":", alpha=0.7)
ax3D.set_xscale("log")
ax3D.set_xlabel("Viscosity (cP)"); ax3D.set_ylabel("Systematic Error (%)")
ax3D.grid(False)
style_panel(ax3D)

# E: post-calibration CV vs viscosity at matched 3.0 g target
for proto in mech_order:
    rows = metrics3g[proto]
    if not rows:
        continue
    viscs = [r[1] for r in rows]; cvs = [r[3] for r in rows]
    ax3E.plot(viscs, cvs, mech_markers[proto] + "-", color=mech_colors[proto], label=mech_names[proto], linewidth=2.2, markersize=9)
ax3E.axhline(2, color=G_GREEN, linewidth=1.3, linestyle=":", alpha=0.8)
ax3E.set_xscale("log")
ax3E.set_xlabel("Viscosity (cP)"); ax3E.set_ylabel("CV (%)")
ax3E.grid(False)
style_panel(ax3E)

label_panels(fig3, [("A", ax3A, 0, 0), ("B", ax3B, 0, 2), ("C", ax3C, 0, 4), ("D", ax3D, 1, 1), ("E", ax3E, 1, 3)])
plt.savefig("../figures/main/fig4_dispensing_accuracy.svg", format="svg", bbox_inches="tight", dpi=300)
plt.savefig("../figures/main/fig4_dispensing_accuracy.png", bbox_inches="tight", dpi=300)
plt.show()
print("Saved: figure3_dispensing_accuracy.svg")

## Figure 5 - Precision and throughput (Section 3.3)

In [ ]:
target_levels = [0.5, 1.0, 3.0, 5.0]

fig4 = plt.figure(figsize=(22, 13))
gs4 = gridspec.GridSpec(2, 3, figure=fig4, hspace=0.38, wspace=0.38)
ax4A = fig4.add_subplot(gs4[0, 0])
ax4B = fig4.add_subplot(gs4[0, 1])
ax4C = fig4.add_subplot(gs4[0, 2])
ax4D = fig4.add_subplot(gs4[1, 0])
ax4E = fig4.add_subplot(gs4[1, 1])
ax4F = fig4.add_subplot(gs4[1, 2])

# A: distributions of relative error at 12,500 cP across the four shared target masses
group_step = len(mech_order) + 1.5
for t_idx, target_vol in enumerate(target_levels):
    group_start = t_idx * group_step
    ax4A.hlines(0, group_start - 0.4, group_start + len(mech_order) - 1 + 0.4, color="gray", linestyle="--", linewidth=1.1, alpha=0.5)
    for p_idx, proto in enumerate(mech_order):
        d = proto_data[proto]
        test = d[(d["Data Type"] == "Test") & (d["Fluid"] == "Bluesil_47V12500") & (np.isclose(d["Volume (g)"], target_vol))]
        if test.empty:
            continue
        vals = []
        for _, row in test.iterrows():
            for c in run_cols:
                if pd.notna(row[c]):
                    vals.append(row[c])
        vals = np.array(vals)
        rel_err = (vals - target_vol) / target_vol * 100
        pos = group_start + p_idx
        ax4A.boxplot([rel_err], positions=[pos], widths=0.6, patch_artist=True, notch=False, showfliers=False,
                     medianprops=dict(color="black", linewidth=1.8),
                     boxprops=dict(facecolor=mech_colors[proto], alpha=0.7, linewidth=1.3),
                     whiskerprops=dict(linewidth=1.3), capprops=dict(linewidth=1.3))
        ax4A.scatter(np.random.normal(pos, 0.07, size=len(rel_err)), rel_err, alpha=0.6, s=18,
                     color=mech_colors[proto], edgecolors="black", linewidth=0.5, zorder=5)

group_centers = [i * group_step + (len(mech_order) - 1) / 2 for i in range(len(target_levels))]
ax4A.set_xticks(group_centers)
ax4A.set_xticklabels([f"{v:g}" for v in target_levels])
ax4A.set_xlabel("Target Mass (g)"); ax4A.set_ylabel("Relative Error (%)")
ax4A.grid(True, axis="y", alpha=0.25)
style_panel(ax4A)

# B: CV vs target mass at matched 12,500 cP
for proto in mech_order:
    d = proto_data[proto]
    test = d[(d["Data Type"] == "Test") & (d["Fluid"] == "Bluesil_47V12500")].sort_values("Volume (g)")
    if test.empty:
        continue
    ax4B.plot(test["Volume (g)"], test["cv_pct"], mech_markers[proto] + "-", color=mech_colors[proto], label=mech_names[proto], linewidth=2.2, markersize=9)
ax4B.axhline(2, color=G_GREEN, linewidth=1.3, linestyle=":", alpha=0.8)
ax4B.set_xlabel("Target Mass (g)"); ax4B.set_ylabel("CV (%)")
ax4B.legend(fontsize=LEGEND_FS, frameon=False)
ax4B.grid(False)
style_panel(ax4B)

# C: job time vs target mass at matched 12,500 cP
for proto in mech_order:
    d = proto_data[proto]
    test = d[(d["Data Type"] == "Test") & (d["Fluid"] == "Bluesil_47V12500")].sort_values("Volume (g)")
    test = test[test["Mean Job Time Total (s)"].notna()]
    if test.empty:
        continue
    ax4C.plot(test["Volume (g)"], test["Mean Job Time Total (s)"] / 60, mech_markers[proto] + "-",
              color=mech_colors[proto], label=mech_names[proto], linewidth=2.2, markersize=9)
ax4C.set_xlabel("Target Mass (g)"); ax4C.set_ylabel("Job Time (min)")
ax4C.grid(False)
style_panel(ax4C)

# D: job time vs viscosity at matched 3.0 g target
for proto in mech_order:
    d = proto_data[proto]
    test = d[(d["Data Type"] == "Test") & (np.isclose(d["Volume (g)"], 3.0))].sort_values("Viscosity (cP)")
    test = test[test["Mean Job Time Total (s)"].notna()]
    if test.empty:
        continue
    ax4D.plot(test["Viscosity (cP)"], test["Mean Job Time Total (s)"] / 60, mech_markers[proto] + "-",
              color=mech_colors[proto], label=mech_names[proto], linewidth=2.2, markersize=9)
ax4D.set_xscale("log")
ax4D.set_xlabel("Viscosity (cP)"); ax4D.set_ylabel("Job Time (min)")
ax4D.grid(False)
style_panel(ax4D)

# E: stacked In/Out/Misc time at matched 12,500 cP, 3.0 g
stack_protos = [4, 5, 6]
x_e = np.arange(len(stack_protos))
in_e, out_e, misc_e = [], [], []
for proto in stack_protos:
    d = proto_data[proto]
    sub = d[(d["Data Type"] == "Test") & (d["Fluid"] == "Bluesil_47V12500") & (np.isclose(d["Volume (g)"], 3.0))]
    in_e.append(sub["Mean Job Time In (s)"].mean() / 60 if not sub.empty else 0.0)
    out_e.append(sub["Mean Job Time Out (s)"].mean() / 60 if not sub.empty else 0.0)
    misc_e.append(sub["Mean Job Time Misc (s)"].mean() / 60 if not sub.empty else 0.0)
in_e, out_e, misc_e = np.array(in_e), np.array(out_e), np.array(misc_e)

ax4E.bar(x_e, in_e, width=0.6, color=G_BLUE, label="Intake")
ax4E.bar(x_e, out_e, bottom=in_e, width=0.6, color=G_GREEN, label="Dispense")
ax4E.bar(x_e, misc_e, bottom=in_e + out_e, width=0.6, color=G_RED, label="Misc")
ax4E.set_xticks(x_e)
ax4E.set_xticklabels(["Remote-\nReservoir", "On-Board\nReservoir", "Pressurized\nOn-Board"], fontsize=TICK_FS)
ax4E.set_ylabel("Stacked Time (min)")
ax4E.legend(fontsize=LEGEND_FS, frameon=False)
ax4E.grid(False)
style_panel(ax4E)

# F: case study - actuator step-size effect (Fill-and-Dispense only)
step_types = {
    "Test":      ("Standard (1x)", G_BLUE, "o"),
    "Test (2x)": ("Double Speed (2x)", G_GREEN, "^"),
    "Test (3x)": ("Triple Speed (3x)", G_RED, "s"),
}
for dtype, (label, color, marker) in step_types.items():
    sub = df[(df["Prototype"] == 3) & (df["Data Type"] == dtype) & (df["Fluid"] == "Bluesil_47V12500")].sort_values("Volume (g)")
    if sub.empty:
        continue
    ax4F.plot(sub["Volume (g)"], sub["Mean Job Time Total (s)"] / 60, marker=marker, color=color, label=label, linewidth=2.4, markersize=9)
ax4F.set_xlabel("Target Mass (g)"); ax4F.set_ylabel("Job Time (min)")
ax4F.legend(fontsize=LEGEND_FS, frameon=False)
ax4F.grid(False)
style_panel(ax4F)

legend_handles4 = [Patch(facecolor=mech_colors[p], edgecolor="black", alpha=0.7, label=mech_names[p]) for p in mech_order]
fig4.legend(handles=legend_handles4, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.02), fontsize=LEGEND_FS + 2, frameon=False)

label_panels(fig4, [("A", ax4A, 0, 0), ("B", ax4B, 0, 1), ("C", ax4C, 0, 2), ("D", ax4D, 1, 0), ("E", ax4E, 1, 1), ("F", ax4F, 1, 2)])
plt.savefig("../figures/main/fig5_precision_throughput.svg", format="svg", bbox_inches="tight", dpi=300)
plt.savefig("../figures/main/fig5_precision_throughput.png", bbox_inches="tight", dpi=300)
plt.show()
print("Saved: figure4_precision_throughput.svg")

## Figure 6 - Strategy comparison scorecard (Section 3.4)

Strategy schematics (A-D) above a scorecard (E) whose columns align with them: cell colour is the min-max normalized score across the four strategies (1 = best), cell text the raw value. Accuracy, precision and throughput at the matched condition (Bluesilv12, 14,570 cP).

In [ ]:
# Radar metrics. Accuracy, precision and throughput are evaluated at the matched condition
# (Bluesilv12, 14,570 cP, the only fluid tested with all four strategies):
# - accuracy: volumetric fidelity |1 - a/rho| from the pre-calibration slope a, so the density
#   term that calibration removes does not penalize any strategy
# - precision: mean CV over the four validation targets (0.5, 1.0, 3.0, 5.0 g)
# - throughput: mean job time for a 5.0 g target
# - viscosity range: highest viscosity reliably dispensed in validation runs
MATCHED_VISC = 14570
accuracy, precision, throughput, viscosity_range = {}, {}, {}, {}
for proto in mech_order:
    d = proto_data[proto]
    cal_m = d[(d["Data Type"] == "Calibration") & (d["Viscosity (cP)"] == MATCHED_VISC)].sort_values("Volume (g)")
    test_m = d[(d["Data Type"] == "Test") & (d["Viscosity (cP)"] == MATCHED_VISC)]
    test_p = d[d["Data Type"] == "Test"]

    slope = stats.linregress(cal_m["Volume (g)"], cal_m["mean_dispensed"]).slope
    accuracy[proto] = abs(1 - slope / DENSITY.get(MATCHED_VISC, 0.973))
    precision[proto] = test_m["cv_pct"].mean()
    throughput[proto] = (test_m.loc[np.isclose(test_m["Volume (g)"], 5.0), "Mean Job Time Total (s)"] / 60).mean()
    viscosity_range[proto] = test_p["Viscosity (cP)"].max()

# Qualitative axes, drawn from the design descriptions in S3.1: whether the pen needs
# the CNC stage to fill (CNC independence) and whether its reservoir replenishes itself
# or needs a human between campaigns (refill autonomy), averaged into one autonomy score;
# and mechanical simplicity, an ordinal rank of tubing runs, valve ports and seals.
cnc_independence    = {3: 0.0, 4: 1.0, 5: 1.0, 6: 1.0}
refill_autonomy     = {3: 1.0, 4: 1.0, 5: 0.0, 6: 0.0}
autonomy            = {p: (cnc_independence[p] + refill_autonomy[p]) / 2 for p in mech_order}
mech_simplicity_raw = {3: 1.0, 4: 0.3, 5: 0.65, 6: 0.45}

def norm(d, invert=True, floor=0.08):
    vals = np.array(list(d.values()), dtype=float)
    lo, hi = vals.min(), vals.max()
    out = {}
    for k, v in d.items():
        n = (v - lo) / (hi - lo) if hi > lo else 0.5
        s = 1 - n if invert else n
        out[k] = floor + s * (1 - floor)
    return out

acc_n   = norm(accuracy,          invert=True)
prec_n  = norm(precision,         invert=True)
thr_n   = norm(throughput,        invert=True)
visc_n  = norm(viscosity_range,   invert=False)
auto_n  = norm(autonomy,          invert=False)
simp_n  = norm(mech_simplicity_raw, invert=False)

radar_table = pd.DataFrame({
    "Accuracy |1-a/rho|": accuracy, "Precision mean CV (%)": precision, "Throughput 5 g (min)": throughput,
    "Viscosity range (cP)": viscosity_range, "Autonomy": autonomy, "Simplicity": mech_simplicity_raw,
    "acc_n": acc_n, "prec_n": prec_n, "thr_n": thr_n, "visc_n": visc_n, "auto_n": auto_n, "simp_n": simp_n,
}).rename(index=mech_names)
print(radar_table.round(3).to_string())

# Figure 6: strategy schematics (A-D) above a scorecard (E) whose columns align with them.
# Cell colour = min-max normalized score (1 = best, 0 = worst); cell text = raw value.
from matplotlib.colors import LinearSegmentedColormap

score01 = {
    "acc": norm(accuracy, invert=True, floor=0), "prec": norm(precision, invert=True, floor=0),
    "thr": norm(throughput, invert=True, floor=0), "visc": norm(viscosity_range, invert=False, floor=0),
    "auto": norm(autonomy, invert=False, floor=0), "simp": norm(mech_simplicity_raw, invert=False, floor=0),
}
raw = {"acc": accuracy, "prec": precision, "thr": throughput, "visc": viscosity_range, "auto": autonomy, "simp": mech_simplicity_raw}
fmt = {"acc": "{:.3f}", "prec": "{:.2f}%", "thr": "{:.1f} min", "visc": "{:,.0f} cP", "auto": "{:.1f}", "simp": "{:.2f}"}
rows = [("acc", "Accuracy", "|1 − a/ρ| at 14,570 cP", False),
        ("prec", "Precision", "mean CV, 0.5–5 g", False),
        ("thr", "Throughput", "5 g job time", False),
        ("visc", "Viscosity range", "highest dispensed", False),
        ("auto", "Operational autonomy", "design score", True),
        ("simp", "Mechanical simplicity", "design score", True)]

panel_letters = {3: "A", 4: "B", 5: "C", 6: "D"}
img_paths = {
    3: "../figures/inputs/Image_002_1.tif",
    4: "../figures/inputs/Image_003_1.tif",
    5: "../figures/inputs/Image_004_1.tif",
    6: "../figures/inputs/Image_005_1.tif",
}
score_cmap = LinearSegmentedColormap.from_list("score", ["#F4F7FB", "#9DB7DE", "#2F5DA8", "#17336B"])
INK6 = "#1E2533"

fig5 = plt.figure(figsize=(20, 12.5))
L, R_, gap = 0.165, 0.995, 0.008
n_cols = len(mech_order)
col_w = (R_ - L - (n_cols - 1) * gap) / n_cols
hm_y0, hm_h = 0.065, 0.41
row_h = hm_h / len(rows)

for i, proto in enumerate(mech_order):
    x0 = L + i * (col_w + gap)
    fig5.text(x0 + 0.004, 0.985, panel_letters[proto], fontsize=PANEL_FS + 6, va="top", ha="left", color="black")
    fig5.text(x0 + col_w / 2, 0.985, mech_names[proto], fontsize=21, va="top", ha="center", color=mech_colors[proto], fontweight="bold")
    ax_img = fig5.add_axes([x0, 0.50, col_w, 0.44])
    ax_img.imshow(Image.open(img_paths[proto]).convert("RGBA"))
    ax_img.axis("off")

    # scorecard column for this strategy
    for r, (key, _, _, _) in enumerate(rows):
        y0 = hm_y0 + (len(rows) - 1 - r) * row_h
        v = score01[key][proto]
        ax_c = fig5.add_axes([x0, y0 + 0.003, col_w, row_h - 0.006])
        ax_c.set_xlim(0, 1); ax_c.set_ylim(0, 1); ax_c.axis("off")
        ax_c.add_patch(FancyBboxPatch((0.004, 0.02), 0.992, 0.96, boxstyle="round,pad=0,rounding_size=0.06",
                                      facecolor=score_cmap(v), edgecolor="none", mutation_aspect=col_w / (row_h - 0.006) * 20 / 12.5))
        best = np.isclose(v, 1.0)
        ax_c.text(0.5, 0.5, fmt[key].format(raw[key][proto]), ha="center", va="center", fontsize=21,
                  color="white" if v > 0.55 else INK6, fontweight="bold" if best else "normal")

# row labels
for r, (key, name, metric, qualitative) in enumerate(rows):
    yc = hm_y0 + (len(rows) - 1 - r) * row_h + row_h / 2
    fig5.text(L - 0.008, yc + 0.010, name, fontsize=19, ha="right", va="center", color=INK6,
              fontweight="bold", style="italic" if qualitative else "normal")
    fig5.text(L - 0.008, yc - 0.017, metric, fontsize=15, ha="right", va="center", color="#5B6678")

fig5.text(0.004, hm_y0 + hm_h + 0.012, "E", fontsize=PANEL_FS + 6, va="bottom", ha="left", color="black")
# compact colour key under the row labels
ax_key = fig5.add_axes([0.03, 0.036, 0.11, 0.014])
ax_key.imshow(np.linspace(0, 1, 256)[None, :], cmap=score_cmap, aspect="auto", extent=(0, 1, 0, 1))
ax_key.set_yticks([]); ax_key.set_xticks([0, 1]); ax_key.set_xticklabels(["worst", "best"], fontsize=15)
ax_key.tick_params(length=0, pad=3)
for sp in ax_key.spines.values(): sp.set_visible(False)

plt.savefig("../figures/main/fig6_strategy_comparison.svg", format="svg", dpi=300)
plt.savefig("../figures/main/fig6_strategy_comparison.png", dpi=300)
plt.show()
print("Saved: figure5_mechanism_comparison.svg")


## Figure 7 - Multi-pen tool-changing validation (Section 4.2)

Validation data from the Jubilee tool-changer demonstration: a CNC-mounted Jubilee picks the white Formulator-Pen (Bluesilv12, 14,570 cP) from a parking bar, dispenses into two vials, returns it, then picks the black pen (Siltech60, 37,020 cP) and dispenses into two more vials, over two repeated trials (8 vials total, `Data/VideoData.xlsx`). (A) Dispensed mass per vial against its own target. (B) Job duration per vial, split into CNC travel/tool-swap/taring overhead and fluid-path dispense time. (C) Pressurized-reservoir fill level immediately before and after each dispense, in execution order.

In [ ]:
from matplotlib.lines import Line2D

video_df = pd.read_csv("../data/platform_validation.csv").rename(columns={
    "target_g": "volume_ml", "syringe_fill_before_pct": "form_percent_pre_dispense",
    "syringe_fill_after_pct": "form_percent_post_dispense", "dispense_time_s": "form_time_out_s"})
video_df["formulator_profile_token"] = video_df["pen"].map({"white": "BLUESILV12", "black": "SILTECH60"})
video_df["vial_idx"] = np.arange(1, len(video_df) + 1)
video_df["error_pct"] = (video_df["actual_weight_g"] - video_df["volume_ml"]) / video_df["volume_ml"] * 100
video_df["overhead_s"] = video_df["job_duration_s"] - video_df["form_time_out_s"]
video_df["reservoir_drop_pct"] = video_df["form_percent_pre_dispense"] - video_df["form_percent_post_dispense"]

fluid_colors_v  = {"BLUESILV12": G_BLUE, "SILTECH60": G_RED}
fluid_labels_v  = {"BLUESILV12": "Bluesilv12, 14,570 cP (white pen)", "SILTECH60": "Siltech60, 37,020 cP (black pen)"}
fluid_markers_v = {"BLUESILV12": "o", "SILTECH60": "D"}
bar_colors = [fluid_colors_v[f] for f in video_df["formulator_profile_token"]]

# Top row: platform photos (A-C); bottom row: validation data (D-F)
photo_paths = [f"../figures/inputs/figure7_photo_{k}.png" for k in "ABC"]
photos = [Image.open(p).convert("RGB") for p in photo_paths]
photo_aspects = [im.width / im.height for im in photos]

# Fixed layout in inches so the figure has no outer margin: photos span the full width,
# and the three plot slots below share that width, with D's slot starting at A's left edge
fig_w, photo_gap = 24.0, 0.12
photo_h = (fig_w - 2 * photo_gap) / sum(photo_aspects)
plot_h, slot_gap = 8.0, 0.25
slot_w = (fig_w - 2 * slot_gap) / 3
ax_left, ax_right, ax_bottom, ax_top = 1.45, 0.15, 1.3, 0.6   # axes inset within each plot slot
fig_h = photo_h + plot_h

fig7 = plt.figure(figsize=(fig_w, fig_h))
def add_axes_in(x, y, w, h):
    return fig7.add_axes([x / fig_w, y / fig_h, w / fig_w, h / fig_h])

axes7_photo, x = [], 0.0
for k, im in enumerate(photos):
    w = photo_h * photo_aspects[k]
    ax_p = add_axes_in(x, plot_h, w, photo_h)
    ax_p.imshow(im, aspect="auto")
    ax_p.axis("off")
    ax_p.text(0.02 * photo_aspects[0] / photo_aspects[k], 0.975, "ABC"[k], transform=ax_p.transAxes,
              fontsize=PANEL_FS + 10, va="top", ha="left", fontfamily="Arial", color="black")
    axes7_photo.append(ax_p)
    x += w + photo_gap

axes7_plot = []
for k in range(3):
    sx = k * (slot_w + slot_gap)
    axes7_plot.append(add_axes_in(sx + ax_left, ax_bottom, slot_w - ax_left - ax_right, plot_h - ax_bottom - ax_top))
    fig7.text((sx + 0.05) / fig_w, (plot_h - 0.08) / fig_h, "DEF"[k], fontsize=PANEL_FS + 10,
              va="top", ha="left", fontfamily="Arial", color="black")
ax7D, ax7E, ax7F = axes7_plot

# D: dispensed mass per vial against its own target, colored by pen/fluid
ax7D.bar(video_df["vial_idx"], video_df["actual_weight_g"], color=bar_colors, width=0.6, alpha=0.85, edgecolor="black", linewidth=1.0)
for _, row in video_df.iterrows():
    ax7D.hlines(row["volume_ml"], row["vial_idx"] - 0.32, row["vial_idx"] + 0.32, color="black", linestyle="--", linewidth=1.6)
ax7D.set_xticks(video_df["vial_idx"])
ax7D.set_xticklabels([f"V{i}" for i in video_df["vial_idx"]])
ax7D.set_xlabel("Vial"); ax7D.set_ylabel("Dispensed Mass (g)")
legend_handles_v = [Patch(facecolor=fluid_colors_v[f], edgecolor="black", label=fluid_labels_v[f]) for f in ["BLUESILV12", "SILTECH60"]]
legend_handles_v.append(Line2D([0], [0], color="black", linestyle="--", label="Target"))
ax7D.set_ylim(0, video_df["actual_weight_g"].max() * 1.32)
ax7D.legend(handles=legend_handles_v, fontsize=LEGEND_FS - 1, frameon=False, loc="upper right")
ax7D.grid(False)
style_panel(ax7D)

# E: job duration decomposed into fixed tool-change/positioning overhead vs fluid-dependent dispense time
ax7E.bar(video_df["vial_idx"], video_df["overhead_s"], color="#9e9e9e", width=0.6, label="Travel + tool swap + taring")
ax7E.bar(video_df["vial_idx"], video_df["form_time_out_s"], bottom=video_df["overhead_s"], color=bar_colors, width=0.6, edgecolor="black", linewidth=1.0, label="Dispense")
ax7E.set_xticks(video_df["vial_idx"])
ax7E.set_xticklabels([f"V{i}" for i in video_df["vial_idx"]])
ax7E.set_xlabel("Vial"); ax7E.set_ylabel("Job Duration (s)")
ax7E.set_ylim(0, video_df["job_duration_s"].max() * 1.25)
ax7E.legend(fontsize=LEGEND_FS - 1, frameon=False, loc="upper right")
ax7E.grid(False)
style_panel(ax7E)

# F: reservoir fill level immediately before/after each dispense, in execution order
for fluid in ["BLUESILV12", "SILTECH60"]:
    sub = video_df[video_df["formulator_profile_token"] == fluid]
    for _, row in sub.iterrows():
        ax7F.plot([row["vial_idx"] - 0.15, row["vial_idx"] + 0.15],
                  [row["form_percent_pre_dispense"], row["form_percent_post_dispense"]],
                  marker=fluid_markers_v[fluid], color=fluid_colors_v[fluid], linewidth=2.4, markersize=8)
ax7F.set_xticks(video_df["vial_idx"])
ax7F.set_xticklabels([f"V{i}" for i in video_df["vial_idx"]])
ax7F.set_xlabel("Vial"); ax7F.set_ylabel("Syringe Fill (% of stroke)")
ax7F.grid(False)
style_panel(ax7F)

# Larger text for the plots, which are bigger here than in the other figures
for ax in axes7_plot:
    ax.tick_params(axis="both", labelsize=TICK_FS + 6)
    ax.xaxis.label.set_fontsize(LABEL_FS + 6)
    ax.yaxis.label.set_fontsize(LABEL_FS + 6)
    if ax.get_legend():
        for txt in ax.get_legend().get_texts():
            txt.set_fontsize(LEGEND_FS + 5)

plt.savefig("../figures/main/fig7_toolchanger_validation.svg", format="svg", dpi=300)
plt.savefig("../figures/main/fig7_toolchanger_validation.png", dpi=300)
plt.show()
print("Saved: figure7_toolchanger_validation.svg")